# Mini-GPT desde cero con PyTorch

Versión modular del notebook académico. La implementación vive en `src/minigpt`; aquí recorremos pedagógicamente tokenización, embeddings, Q/K/V, máscara causal, multi-head attention, cross-entropy, entrenamiento, inferencia, temperature, top-k, top-p y atención.

In [ ]:
import sys
from pathlib import Path

import torch

from minigpt.config import ModelConfig, TrainingConfig
from minigpt.data import encode_and_split, get_batch
from minigpt.device import seed_everything, select_device
from minigpt.generation import generate_text
from minigpt.model import MiniGPT
from minigpt.tokenizer import CharacterTokenizer
from minigpt.training import train_model
from minigpt.visualization import plot_attention, plot_losses

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
device = select_device("auto")
seed_everything(1337)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

## 1. Tokenización por caracteres

Cada carácter conocido recibe un ID. Es menos eficiente que BPE, pero deja visible todo el proceso y mantiene un vocabulario pequeño.

In [ ]:
texto = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme.\n" * 80
tokenizer = CharacterTokenizer.from_text(texto)
muestra = "En un lugar"
ids = tokenizer.encode(muestra)
print("Vocabulario:", tokenizer.vocab_size)
print("IDs:", ids)
print("Reconstrucción:", tokenizer.decode(ids))

## 2. Tarea autoregresiva y batches

`x` contiene una secuencia; `y` contiene la misma secuencia desplazada una posición. El modelo aprende a predecir el siguiente carácter en todas las posiciones simultáneamente.

In [ ]:
train_tokens, val_tokens = encode_and_split(texto, tokenizer)
x, y = get_batch("train", train_tokens, val_tokens, 4, 16, device)
print(x.shape, y.shape, x.dtype)
print(tokenizer.decode(x[0].tolist()))
print(tokenizer.decode(y[0].tolist()))

## 3. Embeddings, Q/K/V y máscara causal

El modelo suma embeddings de token y posición. Cada cabeza proyecta esa representación en Query, Key y Value. Los scores `Q @ Kᵀ / sqrt(d)` reciben una máscara triangular antes del softmax, de modo que ninguna posición atiende al futuro. Varias cabezas se concatenan; después actúa el feed-forward. Los bloques usan LayerNorm pre-norm y residuales.

In [ ]:
model_config = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    n_embed=32,
    n_head=4,
    n_layer=2,
    block_size=16,
    dropout=0.1,
)
model = MiniGPT(model_config).to(device)
logits, loss = model(x, y)
print("Logits:", logits.shape)
print("Cross-entropy:", float(loss))
print("Parámetros:", f"{model.count_parameters():,}")

## 4. Entrenamiento y checkpoint

El siguiente smoke run usa AdamW durante pocas iteraciones. Para Don Quijote usa la CLI con los perfiles `demo`, `academic` o `cpu_light`.

In [ ]:
training_config = TrainingConfig(
    batch_size=4,
    learning_rate=1e-3,
    max_iters=3,
    eval_interval=1,
    eval_iters=1,
)
history, best_val_loss = train_model(
    model,
    tokenizer,
    train_tokens,
    val_tokens,
    model_config,
    training_config,
    project_root / "runs" / "notebook-smoke",
    device,
)
print("Mejor val loss:", best_val_loss)

In [ ]:
plot_losses(history)

## 5. Inferencia: greedy, temperature, top-k y top-p

La generación es autoregresiva: se recorta el contexto, se toman los logits del último token, se filtra la distribución y se muestrea un carácter. Temperature controla la dispersión; top-k conserva `k` candidatos y top-p conserva el núcleo de probabilidad acumulada.

In [ ]:
print(generate_text(model, tokenizer, "En un ", 40, device, greedy=True))
print(generate_text(model, tokenizer, "En un ", 40, device, temperature=0.8, top_k=10))
print(generate_text(model, tokenizer, "En un ", 40, device, temperature=0.9, top_p=0.9))

## 6. Visualización de atención

El triángulo superior debe permanecer en cero: representa posiciones futuras prohibidas por la máscara causal.

In [ ]:
plot_attention(model, tokenizer, "En un lugar", layer=0, head=0, device=device)

## 7. Qué comparte y qué no comparte con GPT-4

Este proyecto comparte arquitectura decoder-only, predicción del siguiente token, atención causal y generación autoregresiva. No comparte escala, corpus, infraestructura, alineamiento ni capacidades. Es una implementación educativa, no un sustituto de un LLM comercial.